In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from textwrap import fill
from transformers import pipeline
from IPython.display import HTML

classifier = pipeline(
    "sentiment-analysis",
    model="nlptown/bert-base-multilingual-uncased-sentiment"
)

texts = [
    "Entrega rápida e produto com ótima qualidade.",
    "A experiência no site foi confusa e o checkout travou.",
    "Gostei bastante do atendimento e da embalagem.",
    "O produto veio diferente da descrição e precisei devolver.",
    "Navegação simples, compra fácil e prazo cumprido.",
    "Demorou muito para chegar e o suporte não ajudou."
]

results = classifier(texts)

labels = []
scores = []

for r in results:
    label = r["label"]
    score = r["score"]

    stars = int(label[0])

    if stars >= 4:
        labels.append("Positivo")
    elif stars <= 2:
        labels.append("Negativo")
    else:
        labels.append("Neutro")

    scores.append(score)

for text, label, score, result in zip(texts, labels, scores, results):
    print(f"Texto: {text}")
    print(f"Resultado original: {result}")
    print(f"Classificação convertida: {label} ({score:.1%})")
    print("-" * 60)

frames_per_text = 18
n_texts = len(texts)
total_frames = n_texts * frames_per_text

fig = plt.figure(figsize=(11, 6), facecolor="white")
gs = fig.add_gridspec(1, 2, width_ratios=[1.6, 1])

ax_text = fig.add_subplot(gs[0, 0])
ax_bar = fig.add_subplot(gs[0, 1])

plt.subplots_adjust(top=0.82, bottom=0.16, wspace=0.32)

TITLE_SIZE = 22
SECTION_SIZE = 17
TEXT_SIZE = 18
INFO_SIZE = 14
AXIS_SIZE = 14
PRED_SIZE = 16
TICK_SIZE = 14

NEG_COLOR = "#ff3131"
POS_COLOR = "#32cd32"
NEU_COLOR = "#b0b0b0"

def draw_frame(frame):
    ax_text.clear()
    ax_bar.clear()

    idx = frame // frames_per_text
    local_frame = frame % frames_per_text

    text = texts[idx]
    label = labels[idx]
    score = scores[idx]

    fig.suptitle(
        "Da linguagem natural à classificação de sentimento",
        fontsize=TITLE_SIZE,
        fontweight="bold",
        y=0.95
    )

    ax_text.set_xlim(0, 1)
    ax_text.set_ylim(0, 1)
    ax_text.axis("off")

    ax_text.text(
        0.02, 0.92,
        "Review / comentário",
        fontsize=SECTION_SIZE,
        fontweight="bold",
        ha="left",
        va="top"
    )

    wrapped = fill(text, width=34)

    reveal_ratio = min(1.0, (local_frame + 1) / 8) if local_frame < 8 else 1.0
    n_chars = max(1, int(len(wrapped) * reveal_ratio))
    shown_text = wrapped[:n_chars]

    ax_text.text(
        0.02, 0.72,
        shown_text,
        fontsize=TEXT_SIZE,
        ha="left",
        va="top",
        linespacing=1.45,
        bbox=dict(
            boxstyle="round,pad=0.55",
            fc="#f7f7f7",
            ec="#d9d9d9"
        )
    )

    ax_text.text(
        0.02, 0.18,
        "Modelos de NLP podem transformar texto\n em sinal analítico"
        "para apoiar triagem,\n monitoramento e decisão.",
        fontsize=INFO_SIZE,
        ha="left",
        va="top"
    )

    ax_bar.set_xlim(0, 1)
    ax_bar.set_ylim(-0.5, 1.5)

    bar_progress = 0.0 if local_frame < 7 else min(1.0, (local_frame - 6) / 6)
    score_shown = score * bar_progress

    if label == "Positivo":
        probs = [0, score_shown]
        bar_colors = [NEG_COLOR, POS_COLOR]
    elif label == "Negativo":
        probs = [score_shown, 0]
        bar_colors = [NEG_COLOR, POS_COLOR]
    else:
        probs = [0.5 * bar_progress, 0.5 * bar_progress]
        bar_colors = [NEU_COLOR, NEU_COLOR]

    classes = ["Negativo", "Positivo"]

    ax_bar.barh(classes, probs, color=bar_colors)

    ax_bar.set_xlabel(
        "Probabilidade",
        fontsize=AXIS_SIZE,
        labelpad=10
    )

    ax_bar.tick_params(
        axis="both",
        labelsize=TICK_SIZE
    )

    if label == "Positivo":
        predicted_idx = 1
    elif label == "Negativo":
        predicted_idx = 0
    else:
        predicted_idx = None

    if predicted_idx is not None:
        ax_bar.get_yticklabels()[predicted_idx].set_fontweight("bold")

    ax_bar.text(
        0.02, 1.04,
        f"Predição: {label} ({score:.1%})",
        transform=ax_bar.transAxes,
        fontsize=PRED_SIZE,
        fontweight="bold",
        ha="left",
        va="bottom"
    )

    ax_bar.grid(axis="x", alpha=0.2)
    ax_bar.spines["top"].set_visible(False)
    ax_bar.spines["right"].set_visible(False)

anim = FuncAnimation(
    fig,
    draw_frame,
    frames=total_frames,
    interval=260
)

HTML(anim.to_jshtml())

anim.save(
    "nlp_reviews_transformers.gif",
    writer=PillowWriter(fps=4)
)

from google.colab import files
files.download("nlp_reviews_transformers.gif")